In [ ]:
Seria para utilizar 
***csv
-tipo de viajero
-lugar de residencia
-lugar de residencia code
-territorio de las islas que viaja

***apis
moneda
***api 
clima 
lluvia

In [33]:
import pandas as pd

df = pd.read_csv("dataset-ISTAC.csv", sep=",")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168300 entries, 0 to 168299
Data columns (total 15 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   TIME_PERIOD#es                   168300 non-null  str    
 1   TIME_PERIOD_CODE                 168300 non-null  str    
 2   TIPO_VIAJERO#es                  168300 non-null  str    
 3   TIPO_VIAJERO_CODE                168300 non-null  str    
 4   LUGAR_RESIDENCIA#es              168300 non-null  str    
 5   LUGAR_RESIDENCIA_CODE            168300 non-null  str    
 6   TERRITORIO#es                    168300 non-null  str    
 7   TERRITORIO_CODE                  168300 non-null  str    
 8   MEDIDAS#es                       168300 non-null  str    
 9   MEDIDAS_CODE                     168300 non-null  str    
 10  OBS_VALUE                        149441 non-null  float64
 11  CONFIDENCIALIDAD_OBSERVACION#es  0 non-null       float64
 12  NOTAS_OBSERVA

In [2]:
df["OBS_VALUE"].unique()

array([ 1.7120e+01,  4.7030e+01,  7.6900e+00, ...,  5.6780e+03,
        1.1888e+04, -2.7910e+03], shape=(54394,))

In [3]:
print(df.columns.tolist())

# valores distintos de cada columna que no sea numérica
for col in df.columns:
    if col != "OBS_VALUE":
        print(col, "->", df[col].nunique(), "valores distintos")

['TIME_PERIOD#es', 'TIME_PERIOD_CODE', 'TIPO_VIAJERO#es', 'TIPO_VIAJERO_CODE', 'LUGAR_RESIDENCIA#es', 'LUGAR_RESIDENCIA_CODE', 'TERRITORIO#es', 'TERRITORIO_CODE', 'MEDIDAS#es', 'MEDIDAS_CODE', 'OBS_VALUE', 'CONFIDENCIALIDAD_OBSERVACION#es', 'NOTAS_OBSERVACION', 'ESTADO_OBSERVACION#es', 'ESTADO_OBSERVACION_CODE']
TIME_PERIOD#es -> 187 valores distintos
TIME_PERIOD_CODE -> 187 valores distintos
TIPO_VIAJERO#es -> 3 valores distintos
TIPO_VIAJERO_CODE -> 3 valores distintos
LUGAR_RESIDENCIA#es -> 12 valores distintos
LUGAR_RESIDENCIA_CODE -> 12 valores distintos
TERRITORIO#es -> 5 valores distintos
TERRITORIO_CODE -> 5 valores distintos
MEDIDAS#es -> 5 valores distintos
MEDIDAS_CODE -> 5 valores distintos
CONFIDENCIALIDAD_OBSERVACION#es -> 0 valores distintos
NOTAS_OBSERVACION -> 1 valores distintos
ESTADO_OBSERVACION#es -> 1 valores distintos
ESTADO_OBSERVACION_CODE -> 1 valores distintos


In [4]:
df["TIPO_VIAJERO_CODE"].unique()

<ArrowStringArray>
['TURISTA', 'TURISTA_PRINCIPAL', 'TURISTA_SECUNDARIO']
Length: 3, dtype: str

In [5]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 28.05,
    "longitude": -16.57,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean,precipitation_sum,sunshine_duration",
    "timezone": "Atlantic/Canary",
}

r = requests.get(url, params=params)
r.raise_for_status()           # avisa si la petición falla
data = r.json()

clima = pd.DataFrame(data["daily"])
clima["time"] = pd.to_datetime(clima["time"])
clima["mes"] = clima["time"].dt.to_period("M")

mensual = clima.groupby("mes").agg(
    temp_media=("temperature_2m_mean", "mean"),
    lluvia_total=("precipitation_sum", "sum"),
)
print(mensual)

         temp_media  lluvia_total
mes                              
2021-01   16.854839         131.5
2021-02   16.875000          29.7
2021-03   17.480645          18.7
2021-04   18.773333          43.7
2021-05   20.716129           9.8
2021-06   21.963333          15.4
2021-07   25.264516           1.1
2021-08   25.629032           4.0
2021-09   24.740000           4.8
2021-10   23.180645           5.2
2021-11   19.450000          19.1
2021-12   18.732258          25.7
2022-01   18.022581          77.7
2022-02   18.703571          30.7
2022-03   16.809677          58.2
2022-04   18.663333          29.4
2022-05   21.374194           2.4
2022-06   22.130000           0.8
2022-07   26.445161           2.2
2022-08   26.074194           5.2
2022-09   23.896667         145.3
2022-10   23.580645           3.6
2022-11   22.313333           0.4
2022-12   19.883871          70.8
2023-01   18.122581           5.6
2023-02   15.967857          47.6
2023-03   22.096774           1.5
2023-04   21.1

In [6]:
import requests
import pandas as pd

frames = []

for anio in range(2020, 2026):
    url = f"https://api.frankfurter.app/{anio}-01-01..{anio}-12-31"
    r = requests.get(url, params={"from": "EUR", "to": "GBP"})
    r.raise_for_status()
    datos = r.json()["rates"]

    df_anio = pd.DataFrame.from_dict(datos, orient="index")
    frames.append(df_anio)

fx = pd.concat(frames)
fx.index = pd.to_datetime(fx.index)
fx = fx.rename(columns={"GBP": "eur_gbp"})

# Comprobación: filas por año
print(fx.groupby(fx.index.year).size())

# Agrupar por mes
fx["mes"] = fx.index.to_period("M")
fx_mensual = fx.groupby("mes")["eur_gbp"].mean().reset_index()

print(fx_mensual.tail(20))

2019      1
2020    258
2021    259
2022    258
2023    256
2024    257
2025    255
dtype: int64
        mes   eur_gbp
53  2024-05  0.855640
54  2024-06  0.846431
55  2024-07  0.843318
56  2024-08  0.851504
57  2024-09  0.840212
58  2024-10  0.834959
59  2024-11  0.833788
60  2024-12  0.828096
61  2025-01  0.839081
62  2025-02  0.830710
63  2025-03  0.837025
64  2025-04  0.853787
65  2025-05  0.843495
66  2025-06  0.849810
67  2025-07  0.864687
68  2025-08  0.865276
69  2025-09  0.868945
70  2025-10  0.871552
71  2025-11  0.879965
72  2025-12  0.875000


In [34]:
df = df[["TIME_PERIOD#es","TIPO_VIAJERO#es","LUGAR_RESIDENCIA#es","LUGAR_RESIDENCIA_CODE","TERRITORIO#es"]]

In [35]:
df = df.rename(columns={
    "TIME_PERIOD#es": "periodo",
    "TIPO_VIAJERO#es": "tipo _de_viajero",
    "LUGAR_RESIDENCIA#es": "residencia",
    "LUGAR_RESIDENCIA_CODE":"codigo_residencia",
    "TERRITORIO#es":"territorio_viaje"
})

In [30]:
df

,Periodo,tipo _de_viajero,Residencia,Codigo_residencia,Territorio_viaje
0,05/2011,Turista,Reino Unido,GB,Lanzarote
1,05/2011,Turista,Reino Unido,GB,Fuerteventura
2,05/2011,Turista,Reino Unido,GB,Gran Canaria
3,05/2011,Turista,Reino Unido,GB,Tenerife
4,05/2011,Turista,Reino Unido,GB,La Palma
...,...,...,...,...,...
168295,04/2011,Turistas secundarios,Mundo (excluida España),5000_XES,Lanzarote
168296,04/2011,Turistas secundarios,Mundo (excluida España),5000_XES,Fuerteventura
168297,04/2011,Turistas secundarios,Mundo (excluida España),5000_XES,Gran Canaria
168298,04/2011,Turistas secundarios,Mundo (excluida España),5000_XES,Tenerife


In [36]:
print(df.columns.tolist())

['periodo', 'tipo _de_viajero', 'residencia', 'codigo_residencia', 'territorio_viaje']
